<a href="https://colab.research.google.com/github/BilodeauS/TheWalkers_ClassProject_TheWalkers-Group6-/blob/main/figures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Figures
Figure 1: within-person change in sleepiness (KSS) vs theta/alpha ratio. Uses `clean_data/sleepiness_eeg_clean.csv` (56 participants).

In [ ]:
# ============================================================
# Figure: when a participant feels sleepier than usual, does their
# theta/alpha ratio go up?  (within-person change: KSS vs T/A ratio)
# Paste into a Colab cell and run.
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

URL = ("https://raw.githubusercontent.com/BilodeauS/TheWalkers_ClassProject_TheWalkers-Group6-/"
       "refs/heads/main/clean_data/sleepiness_eeg_clean.csv")
df = pd.read_csv(URL)
df["log_ratio"] = np.log10(df["theta_alpha_ratio"])

# 1. Each session relative to that person's own average (this is the "change")
g = df.groupby("participant_id")
df["dKSS"] = df["KSS"] - g["KSS"].transform("mean")
df["dlog"] = df["log_ratio"] - g["log_ratio"].transform("mean")
df["pct"] = (10 ** df["dlog"] - 1) * 100          # % above/below their usual ratio

# 2. Repeated-measures correlation (same as the analysis)
n_sub = df["participant_id"].nunique()
r = (df.dKSS * df.dlog).sum() / np.sqrt((df.dKSS**2).sum() * (df.dlog**2).sum())
dof = len(df) - n_sub - 1
p = 2 * stats.t.sf(abs(r * np.sqrt(dof / (1 - r**2))), dof)
slope = (df.dKSS * df.dlog).sum() / (df.dKSS**2).sum()

# 3. Average change for each step of KSS change (rounded, ends grouped)
df["bin"] = df["dKSS"].round().clip(-2, 2)
binned = df.groupby("bin")["dlog"].agg(["mean", "sem", "count"]).reset_index()
to_pct = lambda v: (10 ** v - 1) * 100

# 4. Plot
BLUE, DARK, INK, MUTED, GRID = "#2a78d6", "#104281", "#0b0b0b", "#52514e", "#e4e3df"
fig, ax = plt.subplots(figsize=(7, 5), dpi=150)
ax.axhline(0, color=GRID, lw=1, zorder=0)
ax.axvline(0, color=GRID, lw=1, zorder=0)

jitter = np.random.default_rng(0).uniform(-0.06, 0.06, len(df))
ax.scatter(df.dKSS + jitter, df.pct, s=28, color=BLUE, alpha=0.35,
           edgecolor="white", linewidth=0.6, label="Individual sessions", zorder=2)

ax.errorbar(binned["bin"], to_pct(binned["mean"]),
            yerr=[to_pct(binned["mean"]) - to_pct(binned["mean"] - binned["sem"]),
                  to_pct(binned["mean"] + binned["sem"]) - to_pct(binned["mean"])],
            fmt="o", ms=9, color=DARK, mec="white", mew=1.5, elinewidth=2, capsize=0,
            label="Average per KSS step (± SE)", zorder=4)

xs = np.linspace(df.dKSS.min(), df.dKSS.max(), 50)
ax.plot(xs, to_pct(slope * xs), color=DARK, lw=2, zorder=3)

ax.set_xlabel("Sleepiness (KSS) compared with the participant's own average", color=INK)
ax.set_ylabel("Theta/alpha ratio compared with\nthe participant's own average (%)", color=INK)
ax.set_title("Feeling sleepier than usual goes with a higher theta/alpha ratio",
             loc="left", fontsize=12, color=INK, pad=22)
ax.text(0, 1.02, f"Within-person change, {n_sub} participants × 3 sessions  ·  "
                 f"repeated-measures r = {r:.2f}, p = {p:.3f}",
        transform=ax.transAxes, fontsize=8.5, color=MUTED)
ax.text(0, -0.16, "← more alert than usual", transform=ax.transAxes,
        ha="left", fontsize=8, color=MUTED)
ax.text(1, -0.16, "sleepier than usual →", transform=ax.transAxes,
        ha="right", fontsize=8, color=MUTED)

for s in ["top", "right"]:
    ax.spines[s].set_visible(False)
for s in ["left", "bottom"]:
    ax.spines[s].set_color(MUTED)
ax.tick_params(colors=MUTED)
ax.legend(frameon=False, fontsize=8.5, loc="upper left")
plt.tight_layout()
plt.savefig("kss_vs_ratio_change.png", dpi=300, bbox_inches="tight")
plt.show()

print(binned.assign(mean_pct=to_pct(binned["mean"]).round(1)))